In [2]:
import pandas as pd
import numpy as np

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("data/Historical Data.csv")

df.head()

,Container_ID,Declaration_Date (YYYY-MM-DD),Declaration_Time,Trade_Regime (Import / Export / Transit),Origin_Country,Destination_Port,Destination_Country,HS_Code,Importer_ID,Exporter_ID,Declared_Value,Declared_Weight,Measured_Weight,Shipping_Line,Dwell_Time_Hours,Clearance_Status
0,97061800,2020-01-01,20:16:40,Import,BE,PORT_30,UG,440890,QLRUBN9,0VKY2BR,372254.40,108.0,106.510,LINE_MODE_10,37.6,Clear
1,85945189,2020-01-01,09:43:49,Import,CN,PORT_40,CA,690722,7JD1S2X,8WDKMC6,375751.20,11352.0,11541.578,LINE_MODE_40,51.7,Clear
2,77854751,2020-01-01,06:15:11,Import,CN,PORT_20,BA,620822,WI9O3I5,4DT3246,5353.02,20.7,20.404,LINE_MODE_40,31.3,Clear
3,46925060,2020-01-01,04:04:20,Import,VN,PORT_40,MN,940350,6LI9721,PKUOG2P,1477645.40,9218.0,8814.252,LINE_MODE_40,11.8,Clear
4,34131149,2020-01-01,03:36:29,Import,VN,PORT_20,LV,71080,RZ871V1,1HMVIVH,6364800.00,24000.0,24880.800,LINE_MODE_10,70.9,Clear


In [5]:
df.shape

(54000, 16)

In [6]:
df.columns

Index(['Container_ID', 'Declaration_Date (YYYY-MM-DD)', 'Declaration_Time',
       'Trade_Regime (Import / Export / Transit)', 'Origin_Country',
       'Destination_Port', 'Destination_Country', 'HS_Code', 'Importer_ID',
       'Exporter_ID', 'Declared_Value', 'Declared_Weight', 'Measured_Weight',
       'Shipping_Line', 'Dwell_Time_Hours', 'Clearance_Status'],
      dtype='object')

In [7]:
df.isnull().sum()

Container_ID                                0
Declaration_Date (YYYY-MM-DD)               0
Declaration_Time                            0
Trade_Regime (Import / Export / Transit)    0
Origin_Country                              0
Destination_Port                            0
Destination_Country                         0
HS_Code                                     0
Importer_ID                                 0
Exporter_ID                                 0
Declared_Value                              0
Declared_Weight                             0
Measured_Weight                             0
Shipping_Line                               0
Dwell_Time_Hours                            0
Clearance_Status                            0
dtype: int64

In [8]:
df.fillna(0, inplace=True)

In [10]:
df["Declaration_Date (YYYY-MM-DD)"] = pd.to_datetime(df["Declaration_Date (YYYY-MM-DD)"])

In [11]:
df["month"] = df["Declaration_Date (YYYY-MM-DD)"].dt.month
df["day"] = df["Declaration_Date (YYYY-MM-DD)"].dt.day
df["weekday"] = df["Declaration_Date (YYYY-MM-DD)"].dt.weekday

In [12]:
df["hour"] = pd.to_datetime(df["Declaration_Time"]).dt.hour

C:\Users\Pratham\AppData\Local\Temp\ipykernel_12940\1104550933.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["hour"] = pd.to_datetime(df["Declaration_Time"]).dt.hour


In [13]:
df.drop(columns=["Declaration_Time"], inplace=True)

In [14]:
df["weight_diff"] = df["Measured_Weight"] - df["Declared_Weight"]

In [15]:
df["weight_ratio"] = df["Measured_Weight"] / (df["Declared_Weight"] + 1)

In [16]:
df["value_per_weight"] = df["Declared_Value"] / (df["Declared_Weight"] + 1)

In [17]:
df["long_dwell"] = (df["Dwell_Time_Hours"] > 48).astype(int)

In [20]:
categorical_cols = [
    "Trade_Regime (Import / Export / Transit)",
    "Origin_Country",
    "Destination_Port",
    "Destination_Country",
    "Shipping_Line"
]

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [22]:
df["Risk_Label"] = df["Clearance_Status"].apply(
    lambda x: 0 if x == "Clear" else 1
)

In [24]:
df.drop(
    columns=[
        "Container_ID",
        "Importer_ID",
        "Exporter_ID",
        "Declaration_Date (YYYY-MM-DD)",
        "Clearance_Status"
    ],
    inplace=True
)

In [25]:
df.head()

,Trade_Regime (Import / Export / Transit),Origin_Country,Destination_Port,Destination_Country,HS_Code,Declared_Value,Declared_Weight,Measured_Weight,Shipping_Line,Dwell_Time_Hours,month,day,weekday,hour,weight_diff,weight_ratio,value_per_weight,long_dwell,Risk_Label
0,0,8,24,54,440890,372254.40,108.0,106.510,59,37.6,1,1,2,20,-1.490,0.977156,3415.177982,0,0
1,0,19,28,10,690722,375751.20,11352.0,11541.578,62,51.7,1,1,2,9,189.578,1.016610,33.097084,1,0
2,0,19,21,4,620822,5353.02,20.7,20.404,62,31.3,1,1,2,6,-0.296,0.940276,246.682949,0,0
3,0,114,28,38,940350,1477645.40,9218.0,8814.252,62,11.8,1,1,2,4,-403.748,0.956096,160.282612,0,0
4,0,114,21,35,71080,6364800.00,24000.0,24880.800,59,70.9,1,1,2,3,880.800,1.036657,265.188950,1,0


In [27]:
df.shape

(54000, 19)

In [28]:
df.info

<bound method DataFrame.info of        Trade_Regime (Import / Export / Transit)  Origin_Country  \
0                                             0               8   
1                                             0              19   
2                                             0              19   
3                                             0             114   
4                                             0             114   
...                                         ...             ...   
53995                                         0              19   
53996                                         0              19   
53997                                         0              19   
53998                                         0              19   
53999                                         0             108   

       Destination_Port  Destination_Country  HS_Code  Declared_Value  \
0                    24                   54   440890    3.722544e+05   
1                

In [ ]:
df.to_csv("processed/processed_historical_data.csv", index=False)